# Kelson / Keel — QLoRA training on Kaggle (free T4 / P100)

Reuses your **exact** `train_keel.py` recipe. The cloud only changes the hardware, not the recipe — same base SHA, same seed, same r8/α16 config.

## Setup (once)
1. **Settings → Accelerator → GPU T4 x2** (or P100). **Settings → Internet → On.**
2. Make a Kaggle **Dataset** with these four files from your repo and *Add Input* (right panel):
   `train_keel.py`, `base_identity.json`, `kelson_corpus_v7.jsonl`, `keel_corpus_v7.jsonl`.
3. Run cells top to bottom. Trained adapters land in `/kaggle/working/out` and get zipped for download.

## Keep this notebook + dataset **Private**
Keel is a control. Its weights stay in your own account — never a public notebook, never a live endpoint. Training here is a batch job that emits a file; that's consistent with the offline handling. Don't stand Keel up as a running Space/API.

In [ ]:
# 1. Environment check — confirm GPU + precision
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| bf16 supported:", torch.cuda.is_bf16_supported())
print()
print(">> T4/P100 report bf16=False and will train in fp16.")
print(">> MATCH THIS to your LOCAL run: your train log prints 'bf16' or 'fp16'.")
print(">> Keep a matched pair (Kelson vX + Keel vX) on the SAME precision so the twin comparison stays clean.")

In [ ]:
# 2. Dependencies — recent stack (Olmo-3 needs a current transformers; bitsandbytes for nf4 + PagedAdamW8bit)
!pip -q install -U transformers peft bitsandbytes accelerate
# If training later errors that the 'Olmo3' architecture is unknown, run:
#   !pip -q install -U git+https://github.com/huggingface/transformers

In [ ]:
# 3. Gather your files into the working dir, and cache the base there so reruns skip the ~15GB re-download
import glob, shutil, os
WORK = "/kaggle/working"
os.environ["HF_HOME"] = f"{WORK}/hf"        # base caches here; persists if you 'Save Version'
os.environ["TOKENIZERS_PARALLELISM"] = "false"
want = {"train_keel.py", "base_identity.json"}
found = {}
for p in glob.glob("/kaggle/input/**/*", recursive=True):
    b = os.path.basename(p)
    if b in want or b.endswith("_corpus_v7.jsonl"):
        shutil.copy(p, os.path.join(WORK, b)); found[b] = p
print("copied:", sorted(found))
assert os.path.exists(f"{WORK}/train_keel.py"), "train_keel.py not found — add your dataset via 'Add Input' (right panel)."
os.chdir(WORK)
print("cwd:", os.getcwd(), "| files:", sorted(f for f in os.listdir(WORK) if f.endswith(('.py','.json','.jsonl'))))

In [ ]:
# 4. Train Kelson v7 (primary seed). First run downloads the base (~a few min on Kaggle's pipe), then trains.
!python train_keel.py --corpus kelson_corpus_v7.jsonl --out out/kelson_adapter_v7 --seed 20260715

In [ ]:
# 5. Train Keel v7 (matched control, primary seed)
!python train_keel.py --corpus keel_corpus_v7.jsonl --out out/keel_adapter_v7 --seed 20260715

In [ ]:
# 6. OPTIONAL — the second seed (your two-seed robustness control). Uncomment to run.
# !python train_keel.py --corpus kelson_corpus_v7.jsonl --out out/kelson_adapter_v7_s2 --seed 20260716
# !python train_keel.py --corpus keel_corpus_v7.jsonl  --out out/keel_adapter_v7_s2  --seed 20260716

In [ ]:
# 7. Zip the adapters for download
import shutil, os
shutil.make_archive("/kaggle/working/adapters_v7", "zip", "/kaggle/working/out")
print("zipped ->", os.path.getsize("/kaggle/working/adapters_v7.zip")//1024, "KB at /kaggle/working/adapters_v7.zip")
print("Download it from the right-panel Output tab (or File -> Download).")

## After download
Unzip into your repo so the folders match your local layout:
`out/kelson_adapter_v7 -> kelson/adapter_v7`, `out/keel_adapter_v7 -> keel/adapter_v7`.

Then measure locally (the probes are light — logit/inference only, fine on your 8GB card):
```
python gate_check.py --model olmo3 --label base
python gate_check.py --model olmo3 --adapter kelson/adapter_v7 --label v7
python xm_probe.py  --model olmo3 --adapter keel/adapter_v7 --label keel
python voices.py    --model olmo3 --adapter keel/adapter_v7 --label keel
```

**Quota tips:** ~30 GPU-h/week free. Hit *Stop Session* when a run finishes so idle time doesn't burn quota. `Save Version` keeps the `/kaggle/working/hf` base cache so the next session skips the re-download.